## Load Packages, Silently

In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("anndataR", "ape", "Biostrings", "BSgenome",  "clevr", "cowplot", "GenomicFeatures", "ggplot2", "ggpubr", "ggtree",
                  "gridExtra", "janitor", "nlme", "MAST", "patchwork", "presto", "RColorBrewer", "rhdf5", "Seurat", "stringr", 
                  "tidyverse", "treeio", "TreeTools", "txdbmaker", "vegan", "zellkonverter")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Identify variants in D. pseudoobscura mutant

### Set up Data

In [2]:
## load in vep results
raw_variants <- read_tsv("../workflow/tools/VEP/variant_effect_output.txt", skip = 95) %>%
    janitor::clean_names()

## clean up the column information
all_variants <- raw_variants %>%
    separate_wider_delim(extra, ";", names = c("impact", "distance", "strand", "class", "symbol", "biotype"), 
                         too_few = "align_start", too_many = "drop") %>%
    mutate(biotype = case_when(grepl("BIOTYPE", symbol) ~ symbol,
                               TRUE ~ biotype),
           symbol = case_when(grepl("SYMBOL", class) ~ class,
                              TRUE ~ symbol),
           class = case_when(grepl("VARIANT_CLASS", distance) ~ distance,
                             grepl("VARIANT_CLASS", strand) ~ strand,
                             TRUE ~ class),
           strand = case_when(grepl("STRAND", distance) ~ distance,
                              TRUE ~ strand),
           distance = case_when(!grepl("DISTANCE", distance) ~ NA,
                                TRUE ~ distance)) %>%
    mutate(across(everything(), ~ str_replace(., ".*=", ""))) %>%
    separate_wider_delim(number_uploaded_variation, "_", names = c("chromosome", "position", "change"), too_few = "align_start", too_many = "drop") %>%
    mutate(position = as.integer(position))

## filter for only variants on the XR
variants <- all_variants %>%
    filter(grepl("ChrX", chromosome)) %>%
    filter(grepl("ChrX.1", chromosome) | grepl("ChrX.4", chromosome))

## visualize clean data
head(variants)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 5891958 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (14): #Uploaded_variation, Location, Allele, Gene, Feature, Feature_type...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


chromosome,position,change,location,allele,gene,feature,feature_type,consequence,c_dna_position,⋯,protein_position,amino_acids,codons,existing_variation,impact,distance,strand,class,symbol,biotype
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
ChrX.1,1173,G/A,ChrX.1:1173,A,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,SNV,NA,NA
ChrX.1,8650,A/T,ChrX.1:8650,T,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,SNV,NA,NA
ChrX.1,9376,T/C,ChrX.1:9376,C,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,SNV,NA,NA
ChrX.1,9421,-/T,ChrX.1:9420-9421,T,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,insertion,NA,NA
ChrX.1,9423,A/C,ChrX.1:9423,C,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,SNV,NA,NA
ChrX.1,9441,T/C,ChrX.1:9441,C,-,-,-,intergenic_variant,-,⋯,-,-,-,-,MODIFIER,NA,NA,SNV,NA,NA


In [3]:
## load in SFP data from Garlovsky et al. 2022
sperm_sfp <- read_csv("input/sperm_sfp_garlovsky(2022).csv")
sfp <- read_csv("input/refined_sfp_garlovsky(2022).csv")

## load in orthologs between D. melanogaster and D. pseudoobscura
mel_ortho <- read_delim("../workflow/trinotate/drosoma_pse_mel.txt", col_names = FALSE)
names(mel_ortho) <- c("dpse", "dmel", "paralog", "score")

## load in tau data
tau <- read.csv("../workflow/yang2018_rnaseq/results/tau_tissue_0.50.csv")
testes_genes <- tau %>%
    filter(tau_tissue == "male_gonad") %>%
    pull("gene")

Rows: 7 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): FBgn, NAME
dbl (2): Accessory glands, Testis

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 68 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): FBgn, Symbol, Chromosome
dbl (1): Abundance

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 13772 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): X1, X2, X3
dbl (1): X4

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Check the summary information for variant effect

In [4]:
## count how many variants
nrow(variants)

[1] 306663

In [5]:
## count the number of SNPs
variants %>%
    filter(class == "SNV") %>%
    nrow()

[1] 240546

In [6]:
## count the number of genes with mutations
variants %>%
    pull("symbol") %>%
    na.omit() %>%
    unique() %>%
    length()

[1] 731

In [7]:
## annotate mutation types
variant_type <- variants %>%
    separate_longer_delim(consequence, delim = ",") %>%
    mutate(type = case_when(grepl("UTR", consequence) ~ "UTR",
                            grepl("splice", consequence) ~ "splicing",
                            grepl("intron", consequence) ~ "intron",
                            grepl("frameshift", consequence) ~ "frameshift",
                            grepl("inframe", consequence) ~ "inframe",
                            grepl("start_lost", consequence) | grepl("stop_gained", consequence) ~ "nonsense",
                            grepl("missense", consequence) ~ "missense",
                            grepl("protein_altering", consequence) ~ "protein altering",
                            grepl("stream_gene_variant", consequence) ~ "distal"))

## count how many variants of each type
variant_type_number <- variant_type %>%
    select(type, symbol) %>%
    na.omit() %>%
    unique() %>%
    group_by(type) %>%
    count()
variant_type_number

type,n
<chr>,<int>
UTR,545
distal,730
frameshift,42
inframe,135
intron,487
missense,369
nonsense,8
protein altering,4
splicing,303


In [8]:
## define regulatory sequences
regulator <- c("upstream_gene_variant", "downstream_gene_variant", "3_prime_UTR_variant", "5_prime_UTR_variant")

## count the number of mutations per gene
multi_mutations <- variants %>%
    separate_longer_delim(consequence, ",") %>%
    filter(biotype == "protein_coding" & consequence != "synonymous_variant") %>%
    mutate(mutation_type = case_when(consequence %in% regulator ~ "Regulatory Element",
                             consequence == "intron_variant" ~ "Intron Variant",
                             grepl("splice", consequence) ~ "Splicing Variant",
                             grepl("inframe", consequence) ~ "Inframe InDel",
                             grepl("frameshift", consequence) ~ "Frameshift InDel",
                             consequence == "protein_altering_variant" ~ "Protein Altering",
                             consequence == "stop_gained" ~ "Nonsense Mutation",
                             consequence == "start_lost" ~ "Start Lost",
                             TRUE ~ "Other")) %>%
    group_by(mutation_type, symbol) %>%
    reframe(count = n())
head(multi_mutations)

mutation_type,symbol,count
<chr>,<chr>,<int>
Frameshift InDel,LOC117184369,2
Frameshift InDel,LOC26532021,1
Frameshift InDel,LOC26532600,1
Frameshift InDel,LOC4805581,2
Frameshift InDel,LOC4805583,4
Frameshift InDel,LOC4805584,8


In [9]:
## calculate the limit that 99% of genes are within
cap <- quantile(multi_mutations$count, probs = 0.75)
attributes(cap) <- NULL
cap

[1] 118

In [10]:
## find the highest number of mutations in a single gene
max(multi_mutations$count)

[1] 9615

In [11]:
## find the median number of mutations per gene
median(multi_mutations$count)

[1] 13

### Identify potential mutations of interest

In [12]:
## identify protein altering and nonsense mutations
potential_interest <- variants %>%
    filter(consequence == "protein_altering_variant" | consequence == "stop_gained") %>%
    select(symbol, position, change, consequence) %>%
    unique()
potential_interest

symbol,position,change,consequence
<chr>,<int>,<chr>,<chr>
LOC6901779,645002,-/AGCCAC,protein_altering_variant
LOC4815337,893839,G/C,stop_gained
LOC6902117,1625489,-/TGT,protein_altering_variant
LOC4815785,2706647,G/T,stop_gained
LOC4815774,2949241,C/T,stop_gained
LOC4815426,3468139,-/ACCAGT,protein_altering_variant
LOC6902257,4221843,-/TGCTGC,protein_altering_variant
LOC4815659,4250261,C/A,stop_gained
LOC26532021,4393675,T/G,stop_gained


### Determine if mutations of interest are testis specific and/or sperm proteins

In [13]:
## correct symbol name to shorter name
sperm_sfp$dmel <- c("Idgf3", "Tsf1", "Idgf4", "CG6357", "scpr-C", "NT5E-2", "Pde1c")

## identify the D. pseudoobscura orthologs
sperm_dpse <- sperm_sfp %>%
    inner_join(mel_ortho, by = "dmel") %>%
    select(dpse, NAME) %>%
    rename(sperm_protein = NAME, symbol = dpse)

## correctsymbol name to short name
sfp_x <- sfp %>%
    filter(Chromosome == "X")
sfp_x$dmel <- c("regucalcin", "Tsf1", "CG5162", "Hsc70-3", "Idgf4", "CG18258", "Grp170")

## identify the D. pseudoobscura orthologs
sfp_dpse <- sfp_x %>%
    inner_join(mel_ortho, by = "dmel") %>%
    select(dpse, Symbol) %>%
    rename(sfp = Symbol, symbol = dpse)

## identify expression of genes
tissue_expression <- tau %>%
    select(gene, max_organ, tau_tissue) %>%
    rename(symbol = gene)

In [14]:
## add sperm information to mutation of interest data frame
potential_annotated <- potential_interest %>%
    left_join(sperm_dpse, by = "symbol") %>%
    left_join(sfp_dpse, by = "symbol") %>%
    left_join(tissue_expression, by = "symbol")

potential_annotated

symbol,position,change,consequence,sperm_protein,sfp,max_organ,tau_tissue
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
LOC6901779,645002,-/AGCCAC,protein_altering_variant,NA,NA,female_abdomen,ambiguous
LOC4815337,893839,G/C,stop_gained,NA,NA,female_gonad,ambiguous
LOC6902117,1625489,-/TGT,protein_altering_variant,NA,NA,female_gonad,ambiguous
LOC4815785,2706647,G/T,stop_gained,NA,NA,male_head,ambiguous
LOC4815774,2949241,C/T,stop_gained,NA,NA,male_digestive,ambiguous
LOC4815426,3468139,-/ACCAGT,protein_altering_variant,NA,NA,female_abdomen,ambiguous
LOC6902257,4221843,-/TGCTGC,protein_altering_variant,NA,NA,male_gonad,ambiguous
LOC4815659,4250261,C/A,stop_gained,NA,NA,female_thorax,ambiguous
LOC26532021,4393675,T/G,stop_gained,NA,NA,male_head,male_head


## Characterize gene expression in D. pseudoobscura mutant

### Set up Data

In [15]:
## load in LogFC data
comparison_norm <- read_csv("../workflow/multigenome_rnaseq/results/logfc_normalized.csv")
head(comparison_norm)

New names:
• `` -> `...1`
Rows: 28593 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): gene, comparison, significance, direction, color
dbl (3): ...1, logfc, pvalue

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


...1,gene,comparison,logfc,pvalue,significance,direction,color
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,HCS66_mgp12,sex ratio_reference,-0.7478660,3.116607e-08,significant,hypo,marginally differentially expressed
2,HCS66_mgp12,sex ratio_mutant,-0.6487602,6.394837e-08,significant,hypo,marginally differentially expressed
3,HCS66_mgp12,reference_mutant,0.1097625,9.912583e-01,not significant,hyper,not significant
4,HCS66_mgp11,sex ratio_reference,-1.4785039,1.809080e-08,significant,hypo,marginally differentially expressed
5,HCS66_mgp11,sex ratio_mutant,-0.9880289,7.678211e-07,significant,hypo,marginally differentially expressed
6,HCS66_mgp11,reference_mutant,0.5010003,1.917109e-03,significant,hyper,marginally differentially expressed


In [16]:
### create database for genes from GFF
dpse_ranges <- makeTxDbFromGFF("/home/yahmed-labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/genomic.gff", format = "gff")

### load in fasta of pseudoobscura
dpse_fasta <- readDNAStringSet("/home/yahmed-labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/GCF_009870125.1_UCI_Dpse_MV25_genomic.fna")

Import genomic features from the file as a GRanges object ... 


OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .extract_transcripts_from_GRanges(tx_IDX, gr, mcols0$type, mcols0$ID, :
“some transcripts have no "transcript_id" attribute ==> their name
  ("tx_name" column in the TxDb object) was set to NA”
Warning message in .extract_transcripts_from_GRanges(tx_IDX, gr, mcols0$type, mcols0$ID, :
“the transcript names ("tx_name" column in the TxDb object) imported
  from the "transcript_id" attribute are not unique”
OK



In [17]:
## load in results of Trinotate
ontology <- read_csv("../workflow/trinotate/dpse_goterms.csv") %>%
    select(-1)
head(ontology)

New names:
• `` -> `...1`
Rows: 915220 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): gene, term, category
dbl (1): ...1

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


gene,term,category
<chr>,<chr>,<chr>
HCS66_mgp01,catalytic activity,molecular_function
HCS66_mgp01,cytochrome-c oxidase activity,molecular_function
HCS66_mgp01,transporter activity,molecular_function
HCS66_mgp01,binding,molecular_function
HCS66_mgp01,copper ion binding,molecular_function
HCS66_mgp01,cellular_component,molecular_function


### Perform GO term analysis

In [18]:
## get list of genes in cluster category
up_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hypo") %>%
    pull("gene")
down_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hyper") %>%
    pull("gene")

In [19]:
# ## get list of GO terms
# go_terms <- unique(ontology$term)

# ## Look for GO terms enriched across clusters for each different group

# ## create empty dataframe to populate
# go_compared <- data.frame(go_term = character(),
#                           direction = character(),
#                           go_genes = double(),
#                           go_ratio = double(),
#                           go_pvalue = double())

# for (i in go_terms) {
    
#     ## format dataframe based on mutation type and associated go term
#     go_df <- ontology %>%
#         mutate(signif = case_when(gene %in% up_genes ~ "signif",
#                                   TRUE ~ "nontype"),
#                go = case_when(term == i ~ "go",
#                               TRUE ~ "nogo")) %>%
#         group_by(signif, go) %>%
#         count() %>%
#         pivot_wider(names_from = go, values_from = n) %>%
#         column_to_rownames("signif") %>%
#         mutate_all(replace_na, replace = 0)

#     ## get the number of genes with both mutation type and go term
#     signif_go <- go_df["signif", "go"]

#     ## only perform chi squared if there are more than 0 genes with go term
#     if (signif_go > 0) {        
        
#         ## perform chi-squared test
#         go_chi <- go_df %>%
#             as.matrix() %>%
#             chisq.test()
#         go_pvalue <- go_chi$p.value

#         ## populate dataframe with data
#         go_compared <- add_row(go_compared, go_term = i, 
#                                             direction = "up",
#                                             go_genes = signif_go, 
#                                             go_ratio = signif_go/length(up_genes), 
#                                             go_pvalue = go_pvalue)
        
#     } 

#     ## repeat for adaptive
#     go_df <- ontology %>%
#         mutate(signif = case_when(gene %in% down_genes ~ "signif",
#                                   TRUE ~ "nontype"),
#                go = case_when(term == i ~ "go",
#                               TRUE ~ "nogo")) %>%
#         group_by(signif, go) %>%
#         count() %>%
#         pivot_wider(names_from = go, values_from = n) %>%
#         column_to_rownames("signif") %>%
#         mutate_all(replace_na, replace = 0)
#     signif_go <- go_df["signif", "go"]

#     if (signif_go > 0) {        
#         go_chi <- go_df %>%
#             as.matrix() %>%
#             chisq.test()
#         go_pvalue <- go_chi$p.value
#         go_compared <- add_row(go_compared, go_term = i, 
#                                             direction = "down",
#                                             go_genes = signif_go, 
#                                             go_ratio = signif_go/length(down_genes), 
#                                             go_pvalue = go_pvalue)
#     } 

# }

# ## perform multiple testing correction
# go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

# write.csv(go_compared, "intermediate/pseudoobscura_mutant_go.csv")

In [20]:
# ## identify genes that are differentially expressed and contain enriched GO terms
# mel_ortho %>%
#     rename(gene = dpse) %>%
#     full_join(ontology, by = "gene", relationship = "many-to-many") %>%
#     select(gene, dmel, paralog, term) %>%
#     filter(gene %in% up_genes & term == "structural molecule activity") %>%
#     unique()

### Identify enrichment among differentially expressed genes

#### Identify testes specific enrichment

In [21]:
## chi squared for all differentially expressed genes
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "sex ratio_mutant") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value

[1] 2.085023e-05

In [22]:
## chi squared for different differentially expressed genes
chisq_up_testes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant") %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other"),
           increased = case_when(significance == "significant" & direction == "hypo" ~ "increased",
                                 TRUE ~ "other")) %>%
    group_by(testes, increased) %>%
    count() %>%
    pivot_wider(names_from = increased, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_up_testes$p.value
chisq_down_testes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant") %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other"),
           increased = case_when(significance == "significant" & direction == "hyper" ~ "increased",
                                 TRUE ~ "other")) %>%
    group_by(testes, increased) %>%
    count() %>%
    pivot_wider(names_from = increased, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_down_testes$p.value

[1] 2.428335e-109

[1] 7.451586e-176

In [23]:
## chi squared for all differentially expressed genes with reference, as control
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "reference_mutant") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value
chisq_testes <- comparison_norm %>%
    mutate(testes = case_when(gene %in% testes_genes ~ "testes",
                              TRUE ~ "other")) %>%
    filter(comparison == "sex ratio_reference") %>%
    group_by(testes, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
chisq_testes$p.value

[1] 8.248402e-07

[1] 0.3035263

#### Identify chromosome specific enrichment

In [24]:
## create GRanges object of all of the genes
genes <- genes(dpse_ranges)

## get list of differentially expresed genes
diff_genes <- comparison_norm %>%
    filter(significance == "significant") %>%
    pull("gene")

## get list of genes in each category 
upregulated_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hypo" & significance == "significant") %>%
    pull("gene")
downregulated_genes <- comparison_norm %>%
    filter(comparison == "sex ratio_mutant" & direction == "hyper" & significance == "significant") %>%
    pull("gene")

## clean up fasta names
fasta_names <- names(dpse_fasta) %>%
    as.data.frame() %>%
    rename(original = 1) %>%
    separate_wider_delim(original, delim = ",", names = c("specifics", "scaffold", "sequencing"), cols_remove = FALSE) %>%
    select(-c(scaffold, sequencing)) %>%
    separate_wider_delim(specifics, delim = " ", names = c("name", "genus", "species", "trash1", "strain", "trash2", "chromosome"), 
                         too_many = "merge", too_few = "align_start") %>%
    select(-c(genus, species, trash1, strain)) %>%
    mutate(chromosome = case_when(trash2 == "chromosome" ~ chromosome,
                                  trash2 == "mitochondrion" ~ "mitochondrion",
                                  TRUE ~ "scaffold")) %>%
    select(original, name, chromosome)

## annotate all of the genes with location and expression
gene_characteristics <- genes %>%
    as_tibble() %>%
    select(seqnames, gene_id) %>%
    mutate(diff = case_when(gene_id %in% diff_genes ~ "diff",
                            TRUE ~ "nondiff"),
           dir = case_when(gene_id %in% upregulated_genes ~ "up",
                           gene_id %in% downregulated_genes ~ "down",
                           TRUE ~ "nondiff")) %>%
    rename(name = seqnames) %>%
    left_join(fasta_names, by = "name") %>%
    select(-c(original, name)) %>%
    mutate(x_chr = case_when(chromosome == "X" ~ "x_chr",
                            TRUE ~ "other_chr"))

## find the percentage of genes that are differentially expressed across chromosome
total_genes <- nrow(gene_characteristics)
signif_genes <- gene_characteristics %>%
    filter(diff == "diff") %>%
    nrow()

(signif_genes/total_genes)*100

[1] 42.18009

In [25]:
write_csv(gene_characteristics, "intermediate/gene_chromosome.csv")

In [26]:
## get the percentage of genes on autosomes that are differentially expressed
nonx_genes <- gene_characteristics %>%
    filter(chromosome != "X") %>%
    nrow()
nonx_signif_genes <- gene_characteristics %>%
    filter(chromosome != "X" & diff == "diff") %>%
    nrow()
nonx_down_genes <- gene_characteristics %>%
    filter(chromosome != "X" & dir == "down") %>%
    nrow()

(nonx_signif_genes/nonx_genes)*100
(nonx_down_genes/nonx_genes)*100

[1] 43.51938

[1] 16.25255

In [27]:
## get the percentage of genes on the X chromosome that are differentially expressed
x_genes <- gene_characteristics %>%
    filter(chromosome == "X") %>%
    nrow()
x_signif_genes <- gene_characteristics %>%
    filter(chromosome == "X" & diff == "diff") %>%
    nrow()
x_down_genes <- gene_characteristics %>%
    filter(chromosome == "X" & dir == "down") %>%
    nrow()

(x_signif_genes/x_genes)*100
(x_down_genes/x_genes)*100

[1] 39.80965

[1] 10.71546

In [28]:
## calculate enrichment of differentially expressed genes on X chromosome
x_diff_chi <- gene_characteristics %>%
    select(gene_id, diff, x_chr) %>%
    group_by(diff, x_chr) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
x_diff_chi$p.value

[1] 2.992799e-06

In [29]:
## calculate enrichment of downregulated genes on X chromosome
up_diff_chi <- gene_characteristics %>%
    select(gene_id, dir, x_chr) %>%
    group_by(dir, x_chr) %>%
    filter(dir != "up") %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
up_diff_chi$p.value

[1] 7.862772e-22

In [30]:
## calculate enrichment of upregulated genes on X chromosome
down_diff_chi <- gene_characteristics %>%
    select(gene_id, dir, x_chr) %>%
    group_by(dir, x_chr) %>%
    filter(dir != "down") %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
down_diff_chi$p.value

[1] 0.5474331

#### Identify intersection of testes specificity and X chromosome position

In [31]:
## find if enriched in all differentially expressed genes
all_xtest <- gene_characteristics %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(diff,xtest) %>%
    count() %>%
    pivot_wider(names_from = diff, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
all_xtest$p.value

[1] 2.555237e-133

In [32]:
## find if enriched in down genes
up_xtest <- gene_characteristics %>%
    filter(dir != "up") %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(dir, xtest) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
up_xtest$p.value

down_xtest <- gene_characteristics %>%
    filter(dir != "down") %>%
    mutate(xtest = case_when(x_chr == x_chr & gene_id %in% testes_genes ~ "yes",
                             TRUE ~ "no")) %>%
    group_by(dir ,xtest) %>%
    count() %>%
    pivot_wider(names_from = dir, values_from = n) %>%
    column_to_rownames("xtest") %>%
    as.matrix() %>%
    chisq.test()
down_xtest$p.value

[1] 3.097983e-311

[1] 1.072472e-08

## Associate Genes with parasperm content across obscura Group

### Set up Data

In [33]:
## load in results from CAGEE
credible_df <- read_csv("../workflow/tools/CAGEE/credible_genes.csv") %>%
    mutate(across(everything(), ~ str_replace(., "\\*", "")),
           mutate(across(starts_with("x"), ~ as.numeric(.x))),
           sign = case_when(signif == 14 ~ x_14,
                            signif == 15 ~ x_15,
                            signif == 12 ~ x_12)) %>%
    na.omit()

Rows: 563 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): transcript_id, x_12, x_14, x_15
num (1): signif



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [34]:
## load in gene expression data
raw_reads <- read_tsv("../workflow/tools/obscura_group_reads.tsv") %>%
    janitor::clean_names() %>%
    as.data.frame() %>%
    select(-desc)

## load in tree
tree <- ape::read.tree("../workflow/tools/select_obscura_tree.nwk")

Rows: 16880 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): DESC, Gene_ID
dbl (8): Drosophila_affinis, Drosophila_azteca, Drosophila_bifasciata, Droso...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [35]:
## load in parasperm information
parasperm <- read_csv("../workflow/tools/parasperm_proportion.csv", skip = 1) %>%
    janitor::clean_names() %>%
    mutate(species = str_replace(species, "D. ", "Drosophila_")) %>%
    rename(parasperm = 2) %>%
    select(species, parasperm)

New names:
• `` -> `...5`
Rows: 10 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): species, Category, Remating in days (from Holman et al 2007 Evoluti...
dbl (1): % Parasperm

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [36]:
## load motif results
credible_motifs <- read_tsv("../workflow/tools/STREME/streme_credible/sequences.tsv") %>%
    janitor::clean_names()
credible_targets <- read_tsv("../workflow/tools/STREME/streme_credible/tomtom.tsv") %>%
    janitor::clean_names()

head(credible_motifs)
head(credible_targets)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 13350 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): motif_ID, motif_ALT_ID, seq_ID, seq_Class
dbl (3): motif_P-value, seq_Score, is_holdout?

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 81 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (5): Query_ID, Target_ID, Query_consensus, Target_consensus, Orientation
dbl (5): Optimal_offset, p-value, E-value, q-value, Overlap

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column 

motif_id,motif_alt_id,motif_p_value,seq_id,seq_score,seq_class,is_holdout
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>
1-GGGGGGGGACG,STREME-1,0.0078,LOC13036322,21.15,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6903534,21.15,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC4815230,21.15,fp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC4817549,21.15,fp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC26534045,21.15,fp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6898840,21.15,fp,0


query_id,target_id,optimal_offset,p_value,e_value,q_value,overlap,query_consensus,target_consensus,orientation
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1-GGGGGGGGACG,FBgn0086910,5,4.43841e-07,0.00062981,0.00124231,11,GGGGGGGGACG,GGGGGGGGGGGGGACGTGT,+
1-GGGGGGGGACG,FBgn0086910_2,4,2.06429e-05,0.02929230,0.01643510,11,GGGGGGGGACG,GGGGGGGGGGGGGGG,+
1-GGGGGGGGACG,FBgn0036179,2,2.34871e-05,0.03332820,0.01643510,11,GGGGGGGGACG,GGGGGGGGGGGGGGGATAAT,+
1-GGGGGGGGACG,OTF0481.1,2,2.34871e-05,0.03332820,0.01643510,11,GGGGGGGGACG,GGGGGGGGGGGGGGGATAAT,+
1-GGGGGGGGACG,FBgn0020309,2,3.72195e-05,0.05281440,0.02001810,11,GGGGGGGGACG,GGGGAGGGGGGGGGT,-
1-GGGGGGGGACG,OTF0361.1,0,4.29114e-05,0.06089130,0.02001810,11,GGGGGGGGACG,GGGGGGGGGGG,+


### Characterize credible genes

In [37]:
## get genes with credible changes at nodes with increases in gene expression (without conflicting if present in multiple)
credible_genes <- credible_df %>%
    pull("transcript_id")
length(unique(credible_genes))

[1] 561

In [38]:
## get the direction of the differentially expressed genes
credible_signs <- credible_df %>%
    mutate(direction = case_when(sign > 0 ~ "up",
                                 sign < 0 ~ "down")) %>%
    group_by(direction) %>%
    count()
credible_signs

direction,n
<chr>,<int>
down,540
up,21


In [39]:
## get list of genes with all the same direction
credible_down_genes <- credible_df %>%
    filter(sign < 0) %>%
    pull("transcript_id")
length(credible_down_genes)

[1] 540

### Determine if there is enrichment for branch associated genes on X or in testes specificity

In [40]:
## add additional gene characteristics
add_gene_char <- gene_characteristics %>%
    mutate(credible = case_when(gene_id %in% credible_down_genes ~ "credible",
                                TRUE ~ "other"),
           testes = case_when(gene_id %in% testes_genes ~ "testes",
                              TRUE ~ "other"))
head(add_gene_char)

gene_id,diff,dir,chromosome,x_chr,credible,testes
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
HCS66_mgp01,diff,up,mitochondrion,other_chr,other,other
HCS66_mgp02,diff,up,mitochondrion,other_chr,other,other
HCS66_mgp03,diff,up,mitochondrion,other_chr,other,other
HCS66_mgp04,diff,up,mitochondrion,other_chr,other,other
HCS66_mgp05,diff,up,mitochondrion,other_chr,other,other
HCS66_mgp06,diff,up,mitochondrion,other_chr,other,other


In [41]:
## calculate enrichment of downregulated genes on X chromosome
credible_testes_chi <- add_gene_char %>%
    group_by(credible, testes) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("testes") %>%
    as.matrix() %>%
    chisq.test()
credible_testes_chi$p.value

add_gene_char %>%
    filter(credible == "credible") %>%
    group_by(testes) %>%
    count()

[1] 0

testes,n
<chr>,<int>
other,91
testes,449


In [42]:
tau %>%
    filter(gene %in% credible_down_genes & tau_tissue == "ambiguous") %>%
    group_by(max_organ) %>%
    count()

max_organ,n
<chr>,<int>
female_abdomen,7
female_digestive,4
female_gonad,7
female_rt,3
male_digestive,1
male_gonad,56
male_rt,2
male_thorax,3


In [43]:
## calculate enrichment of downregulated genes on X chromosome
credible_x_chi <- add_gene_char %>%
    group_by(credible, x_chr) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("x_chr") %>%
    as.matrix() %>%
    chisq.test()
credible_x_chi$p.value

add_gene_char %>%
    filter(credible == "credible") %>%
    group_by(chromosome) %>%
    count()

[1] 0.4362217

chromosome,n
<chr>,<int>
2,99
3,58
4,175
5,3
X,204
scaffold,1


### Identify GO terms that are enriched among branch associated genes

In [44]:
# ## Look for GO terms enriched across clusters for each different group

# ## get the list of GO terms
# go_terms <- unique(ontology$term)

# ## create empty dataframe to populate
# go_compared <- data.frame(go_term = character(),
#                           go_genes = double(),
#                           go_ratio = double(),
#                           go_pvalue = double())

# for (i in go_terms) {
    
#     ## format dataframe based on mutation type and associated go term
#     go_df <- ontology %>%
#         mutate(signif = case_when(gene %in% credible_down_genes ~ "signif",
#                                   TRUE ~ "nontype"),
#                go = case_when(term == i ~ "go",
#                               TRUE ~ "nogo")) %>%
#         group_by(signif, go) %>%
#         count() %>%
#         pivot_wider(names_from = go, values_from = n) %>%
#         column_to_rownames("signif") %>%
#         mutate_all(replace_na, replace = 0)

#     ## get the number of genes with both mutation type and go term
#     signif_go <- go_df["signif", "go"]

#     ## only perform chi squared if there are more than 0 genes with go term
#     if (signif_go > 0) {        
        
#         ## perform chi-squared test
#         go_chi <- go_df %>%
#             as.matrix() %>%
#             chisq.test()
#         go_pvalue <- go_chi$p.value

#         ## populate dataframe with data
#         go_compared <- add_row(go_compared, go_term = i, 
#                                             go_genes = signif_go, 
#                                             go_ratio = signif_go/length(credible_down_genes), 
#                                             go_pvalue = go_pvalue)
        
#     } 
# }

# ## perform multiple testing correction
# go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

# write.csv(go_compared, "intermediate/credible_go.csv")

In [45]:
## investigate the genes that drive the enrichment of the nuclear envelope
ontology %>%
    filter(gene %in% credible_down_genes & grepl("nuclear envelope", term)) %>%
    rename(dpse = gene) %>%
    left_join(mel_ortho, by = "dpse")

dpse,term,category,dmel,paralog,score
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC117184964,nuclear envelope,molecular_function,NA,NA,NA
LOC117184964,nuclear envelope organization,molecular_function,NA,NA,NA
LOC117184964,nuclear envelope organization,biological_process,NA,NA,NA
LOC117184964,nuclear envelope,biological_process,NA,NA,NA
LOC117184964,nuclear envelope,cellular_component,NA,NA,NA
LOC117184964,nuclear envelope organization,cellular_component,NA,NA,NA
LOC4812921,nuclear envelope,molecular_function,Patj,n:1,20438
LOC4812921,nuclear envelope,biological_process,Patj,n:1,20438
LOC4812921,nuclear envelope,cellular_component,Patj,n:1,20438


In [46]:
## investigate the genes that drive the enrichment of FFAT motif binding
ontology %>%
    filter(gene %in% credible_down_genes & grepl("FFAT", term)) %>%
    rename(dpse = gene) %>%
    left_join(mel_ortho, by = "dpse")

dpse,term,category,dmel,paralog,score
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC6900207,FFAT motif binding,cellular_component,fan,n:1,NA
LOC6902465,FFAT motif binding,cellular_component,NA,NA,NA
LOC6902496,FFAT motif binding,cellular_component,fan,n:1,30854
LOC6902928,FFAT motif binding,cellular_component,NA,NA,NA


In [47]:
## investigate the genes that drive the enrichment of IRE stress response
ontology %>%
    filter(gene %in% credible_down_genes & grepl("IRE", term)) %>%
    rename(dpse = gene) %>%
    inner_join(mel_ortho, by = "dpse")

dpse,term,category,dmel,paralog,score
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC6900207,IRE1-mediated unfolded protein response,cellular_component,fan,n:1,NA
LOC6902496,IRE1-mediated unfolded protein response,cellular_component,fan,n:1,30854


In [48]:
## investigate the genes associated with ER stress response
### IRE
### PERK
mel_ortho %>%
    filter(dpse %in% credible_down_genes & grepl("parp", dmel, ignore.case = TRUE))

dpse,dmel,paralog,score
<chr>,<chr>,<chr>,<dbl>
LOC6898244,Parp16,1:1,7191


### Identify motifs enriched among branch-associated genes

In [49]:
credible_motifs %>%
    filter(motif_p_value < 0.05 & seq_id %in% credible_down_genes)

motif_id,motif_alt_id,motif_p_value,seq_id,seq_score,seq_class,is_holdout
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>
1-GGGGGGGGACG,STREME-1,0.0078,LOC13036322,21.15,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6903534,21.15,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC117184593,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC26533335,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC117184603,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6901665,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC26532845,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6901481,19.76,tp,0
1-GGGGGGGGACG,STREME-1,0.0078,LOC6901126,18.44,tp,0


## Compare parasperm associated genes with mutant information

In [50]:
## perform a PCA on the reads
pca <- raw_reads %>%
    column_to_rownames("gene_id") %>%
    as.matrix() %>%
    t() %>%
    prcomp()
summary(pca)

## determines genes that contribute to variation
pca_loading <- pca$rotation %>%
    as.data.frame() %>%
    select(PC1, PC2, PC3) %>%
    rownames_to_column("gene") %>%
    pivot_longer(!gene, names_to = "pc", values_to = "loading") %>%
    group_by(pc) %>% 
    arrange(desc(abs(loading)))

## get top 20% of genes that explain variance
var_genes <- pca_loading %>%
    slice(1:(length(unique(pca_loading$gene))/5)) %>%
    pull("gene") %>%
    unique()

## reformat reads so each gene is a row and each species is a column
reads <- raw_reads %>%
    filter(gene_id %in% var_genes) %>% ## filter for only the most variable genes
    mutate_at(vars(-gene_id), as.double) %>%
    column_to_rownames("gene_id") %>%
    t() %>%
    as.data.frame() %>%
    rownames_to_column("species") %>%
    mutate(species = str_replace(species, "drosophila", "Drosophila")) %>%
    column_to_rownames("species")

## get list of species
species <- rownames(reads)

Importance of components:
                             PC1       PC2       PC3       PC4       PC5
Standard deviation     4.939e+05 4.470e+05 4.065e+05 2.969e+05 2.914e+05
Proportion of Variance 2.814e-01 2.304e-01 1.906e-01 1.017e-01 9.792e-02
Cumulative Proportion  2.814e-01 5.119e-01 7.025e-01 8.042e-01 9.021e-01
                             PC6       PC7       PC8
Standard deviation     2.456e+05 1.567e+05 5.255e-10
Proportion of Variance 6.958e-02 2.833e-02 0.000e+00
Cumulative Proportion  9.717e-01 1.000e+00 1.000e+00

In [51]:
## combine differential expression and parasperm data
annotated_logfc <- colnames(reads) %>%
    as.data.frame() %>%
    rename(gene = 1) %>%
    left_join(comparison_norm, by = "gene") %>%
    filter(comparison == "sex ratio_mutant") %>%
    mutate(logfc = -(logfc),
           credible = case_when(gene %in% credible_down_genes ~ "credible",
                                TRUE ~ "non-credible"),
           significance = case_when(is.na(significance) ~ "not significant",
                                    TRUE ~ significance),
           color = case_when(is.na(color) ~ "not significant",
                             TRUE ~ color),
           direction = case_when(significance == "significant" & logfc < 0 ~ "hypo",
                                 significance == "significant" & logfc > 0 ~ "hyper",
                                 TRUE ~ "not significant"),
           testes_specificity = case_when(gene %in% testes_genes ~ "testes",
                                          TRUE ~ "other"),
           x = case_when(gene %in% x_genes ~ "x",
                         TRUE ~ "other")) %>%
    unique()

### Identify enrichment of parasperm associated genes among differentially expressed genes

In [52]:
## compare number of significant genes to those with credible and adaptive
credible_logfc <- annotated_logfc %>%
    group_by(credible, significance) %>%
    count() %>%
    pivot_wider(names_from = significance, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix()

## perform chi squared test
chisq.test(credible_logfc)$p.value

[1] 3.660479e-08

In [53]:
## now look at credible, separating increase and decrease
credible_up_mx <- annotated_logfc %>%
    filter(direction != "hypo") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix()
credible_up_chi <- chisq.test(credible_up_mx)
credible_up_chi$p.value

credible_down_mx <- annotated_logfc %>%
    filter(direction != "hyper") %>%
    group_by(credible, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("credible") %>%
    mutate_all(replace_na, replace = 0) %>%
    as.matrix()
credible_down_chi <- chisq.test(credible_down_mx)
credible_down_chi$p.value

[1] 8.05134e-07

[1] 4.412681e-28

In [54]:
## identify the expression of credible genes
annotated_logfc %>%
    filter(credible == "credible") %>%
    group_by(direction) %>%
    count()

direction,n
<chr>,<int>
hyper,20
hypo,221
not significant,112


In [55]:
## get the list of genes whose increase in expression are associated with parasperm content
credible_up_genes <- credible_genes[!(credible_genes %in% credible_down_genes)]
length(credible_up_genes)

## determine the direction of expression of these genes in the mutant
annotated_logfc %>%
    filter(gene %in% credible_up_genes) %>%
    group_by(direction) %>%
    count()

[1] 21

direction,n
<chr>,<int>
hyper,1
hypo,6
not significant,7


In [56]:
## identify the one gene with differential expression
credible_upup <- annotated_logfc %>%
    filter(gene %in% credible_up_genes & direction == "hyper") %>%
    pull("gene")

## characterize the one gene with increased expression
annotated_logfc %>%
    filter(gene %in% credible_up_genes & direction == "hyper")
mel_ortho %>%
    filter(dpse == credible_upup)
ontology %>%
    filter(gene == credible_upup)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
LOC6897697,5435,sex ratio_mutant,0.3634016,0.0006169901,significant,hyper,marginally differentially expressed,non-credible,testes,other


dpse,dmel,paralog,score
<chr>,<chr>,<chr>,<dbl>


gene,term,category
<chr>,<chr>,<chr>


### Identify GO terms associated with branch associated genes that are downregulated in the mutant

In [57]:
# ## look for enriched GO terms in for credible, up-regulated genes

# ## get the list of GO terms
# go_terms <- unique(ontology$term)

# ## get list of credible, up-regulated genes
# downcred_genes <- annotated_logfc %>%
#     filter(credible == "credible" & direction == "hypo") %>%
#     pull("gene")

# ## create empty dataframe to populate
# go_compared <- data.frame(go_term = character(),
#                           go_genes = double(),
#                           go_ratio = double(),
#                           go_pvalue = double())

# for (i in go_terms) {    
    
#     ## format dataframe based on associated go term
#     go_df <- ontology %>%
#         mutate(signif = case_when(gene %in% downcred_genes ~ "signif",
#                                     TRUE ~ "nontype"),
#                go = case_when(term == i ~ "go",
#                               TRUE ~ "nogo")) %>%
#         group_by(signif, go) %>%
#         count() %>%
#         pivot_wider(names_from = go, values_from = n) %>%
#         column_to_rownames("signif") %>%
#         mutate_all(replace_na, replace = 0)

#     ## get the number of genes with go term
#     signif_go <- go_df["signif", "go"]

#     ## only perform chi squared if there are more than 0 genes with go term
#     if (signif_go > 0) {        
        
#         ## perform chi-squared test
#         go_chi <- go_df %>%
#             as.matrix() %>%
#             chisq.test()
#         go_pvalue <- go_chi$p.value

#         ## populate dataframe with data
#         go_compared <- add_row(go_compared, go_term = i, 
#                                             go_genes = signif_go, 
#                                             go_ratio = signif_go/length(downcred_genes), 
#                                             go_pvalue = go_pvalue)
        
#     }    
# }

# ## perform multiple testing correction
# go_compared$pvalue_corrected <- p.adjust(go_compared$go_pvalue, method = "BH")

# write.csv(go_compared, "intermediate/credible_down_go.csv")

In [58]:
## investigate the genes that drive the enrichment of specific GO terms
ontology %>%
    filter(gene %in% credible_down_genes & gene %in% down_genes & grepl("immune", term)) %>%
    rename(dpse = gene) %>%
    left_join(mel_ortho, by = "dpse")

dpse,term,category,dmel,paralog,score
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC6896671,regulation of innate immune response,cellular_component,CanA1,1:1,18195
LOC6896671,regulation of immune response,cellular_component,CanA1,1:1,18195
LOC6897740,immune response,molecular_function,AGO2,n:1,NA
LOC6897740,immune response,biological_process,AGO2,n:1,NA
LOC6897740,immune response,cellular_component,AGO2,n:1,NA
LOC6897740,RNAi-mediated antiviral immune response,cellular_component,AGO2,n:1,NA
LOC6897740,innate immune response,cellular_component,AGO2,n:1,NA
LOC6901659,immune response,molecular_function,NA,NA,NA
LOC6901659,immune response,biological_process,NA,NA,NA


## Analyze single cell data set

### Load data

In [59]:
## load in single-cell RNA-seq results as singlecell experiment
sce_sce <- readH5AD("/home/yahmed-labshare/scratch/pseudoobscura_testis_scRNASeq/pseudo_testis_scRNAseq.h5ad")

# convert single cell experiment to Seurat
sce <- as.Seurat(sce_sce, counts = "counts", data = "X")

Warning message:
“The names of these selected var columns have been modified to match R
conventions: '2' -> 'X2'”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from X_diffmap_ to Xdiffmap_”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from X_draw_graph_fa_ to Xdrawgraphfa_”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from X_pca_ to Xpca_”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from X_tsne_ to Xtsne_”
Warning message:
“Keys should be one or more alphanumeric char

In [60]:
## create data frame of genes from He et al. 2026
dmel_he_testis_genes <- data.frame(stage = c("GSC/Spermatogonia", "GSC/Spermatogonia", "GSC/Spermatogonia", "GSC/Spermatogonia", "GSC/Spermatogonia",
                                            "Early Spermatocytes", "Early Spermatocytes", "Early Spermatocytes",
                                            "Late Spermatocytes", "Late Spermatocytes", "Late Spermatocytes",
                                            "Spermatid", "Spermatid",
                                            "Cyst", "Cyst",
                                            "Hub", "Epithelial", "Epithelial"),
                                   dmel = c("AGO3", "aub", "bam", "vas", "aly", "can", "sa", "CycB", "twe", "bol", "soti", "wa-cup", "f-cup",
                                            "tj", "eya", "Fas3", "MtnA", "Hsp23"))

## get the D. pseudoobscura counterpart for each gene
he_testis_genes <- mel_ortho %>%
    select(dmel, dpse) %>%
    right_join(dmel_he_testis_genes, by = "dmel")
head(he_testis_genes)

dmel,dpse,stage
<chr>,<chr>,<chr>
f-cup,LOC6897136,Spermatid
soti,LOC4801948,Late Spermatocytes
wa-cup,LOC4802953,Spermatid
bam,LOC4803214,GSC/Spermatogonia
CycB,LOC4803852,Early Spermatocytes
AGO3,LOC26534190,GSC/Spermatogonia


In [61]:
## create data frame of genes from Messer et al. 2026
dmel_messer_testis_genes <- data.frame(stage = c("GSC-SG", "SG", "SG", "SG",
                                                 "SC", "SC", "SC", "SC", "SC", "SC", "SC", "SC",
                                                 "ST", "ST", "ST"),
                                       dmel = c("esg", "shg", "vas", "aly", 
                                                "wuc", "ProtA", "boule", "TGIF", "kmg", "S-Lap5", "S-Lap7", "S-Lap8",
                                                "f-cup", "wa-cup", "p-cup"))

## get the D. pseudoobscura counterpart for each gene
messer_testis_genes <- mel_ortho %>%
    select(dmel, dpse) %>%
    right_join(dmel_messer_testis_genes, by = "dmel") %>%
    na.omit()
messer_testis_genes

dmel,dpse,stage
<chr>,<chr>,<chr>
f-cup,LOC6897136,ST
wa-cup,LOC4802953,ST
S-Lap5,LOC6898081,SC
S-Lap8,LOC4803787,SC
wuc,LOC4804758,SC
S-Lap7,LOC4805161,SC
S-Lap5,LOC4805168,SC
shg,LOC6898987,SG
esg,LOC4816405,GSC-SG


### Process single cell data

In [62]:
# ## find genes that vary between cells
# sce <- FindVariableFeatures(sce, selection.method = "vst", nfeatures = 2000)

# ## scale data
# gene_list <- rownames(sce)
# scaled_sce <- ScaleData(sce, features = gene_list)

# ## run PCA on counts (pca saved in sce[["pca"]])
# pca_sce <- RunPCA(scaled_sce, features = VariableFeatures(object = scaled_sce))

# ## find clusters, based on the first 30 PCs
# cluster_sce <- FindNeighbors(pca_sce, dims = 1:30)
# cluster_sce <- FindClusters(cluster_sce, resolution = 0.75)

# ## run UMAP and annotate with these clusters
# cluster_sce <- RunUMAP(cluster_sce, dims = 1:30)

# ## save the names of the clusters
# cluster_names <- c("Late EuC (0)", "G (1)", "Late EuC (2)", "Late PaC (3)", "Early EuC (4)", "Late PaC (5)", "Epi (6)", "Late EuC (7)", "Early PaC (8)", "Early EuC (9)", "Early EuC (10)", 
#                    "Early EuC (11)", "Early PaC (12)", "G (13)", "EuS (14)", "Early PaC (15)", "Early EuC (16)", "Late PaC (17)", "PaS (18)", "Late EuC (19)", "Early PaC (20)")
# names(cluster_names) <- levels(cluster_sce)

# ## annotate the clusters
# annotated_sce <- RenameIdents(cluster_sce, cluster_names)

# ## reannotated clusters based on cell type
# cluster_names <- c("Late EuC", "G", "Late EuC", "Late PaC", "Early EuC", "Late PaC", "Epi", "Late EuC", "Early PaC", "Early EuC", "Early EuC", 
#                    "Early EuC", "Early PaC", "G", "EuS", "Early PaC", "Early EuC", "Late PaC", "PaS", "Late EuC", "Early PaC")
# names(cluster_names) <- levels(cluster_sce)

# ## annotate the clusters
# annotated_down_sce <- RenameIdents(cluster_sce, cluster_names)

# ## find markers for the annotated cluster
# sce_markers <- FindAllMarkers(object = annotated_down_sce, only.pos = TRUE, return.thresh = 0.01)

# ## save file 
# write_csv(sce_markers, "intermediate/pseudoobscura_testis_markers.csv")

# ## get the cell stage 
# cell_stages <- Idents(annotated_down_sce) %>%
#     as.data.frame() %>%
#     pull(1)

# ## add metadata column for cell stage
# annotated_down_sce$stage <- cell_stages

# ## save object
# saveRDS(annotated_down_sce, "intermediate/processed_sce.rds")

# ## find differentially expressed genes between eusperm and parasperm for different cell types
# early_c_logfc <- FindMarkers(annotated_down_sce, ident.1 = "Early EuC", ident.2 = "Early PaC", test.use = "MAST") %>%
#     mutate(stage = "early_spermatocyte")
# late_c_logfc <- FindMarkers(annotated_down_sce, ident.1 = "Late EuC", ident.2 = "Late PaC", test.use = "MAST") %>%
#     mutate(stage = "late_spermatocyte")
# sperm_logfc <- FindMarkers(annotated_down_sce, ident.1 = "EuS", ident.2 = "PaS", test.use = "MAST") %>%
#     mutate(stage = "sperm")

# ## combine into data set
# logfc_morph <- rbind(early_c_logfc, late_c_logfc, sperm_logfc) %>%
#     rownames_to_column("gene")
# write_csv(logfc_morph, "intermediate/logfc_morph_stage.csv")

### Identify genes that differ between sperm morphs for each stage of development

In [63]:
## load processed single cell experiment object
sce <- readRDS("intermediate/processed_sce.rds")

## load in logFC between stages data
logfc_morph <- read_csv("intermediate/logfc_morph_stage.csv")

Rows: 29336 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): gene, stage
dbl (5): p_val, avg_log2FC, pct.1, pct.2, p_val_adj

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [64]:
## get the list of genes that are significantly differentially expressed among each stage
ec_eu_genes <- logfc_morph %>%
    filter(stage == "early_spermatocyte" & p_val_adj < 0.01 & avg_log2FC > 0) %>%
    pull("gene")
lc_eu_genes <- logfc_morph %>%
    filter(stage == "late_spermatocyte" & p_val_adj < 0.01 & avg_log2FC > 0) %>%
    pull("gene")
s_eu_genes <- logfc_morph %>%
    filter(stage == "late_spermatocyte" & p_val_adj < 0.01 & avg_log2FC > 0) %>%
    pull("gene")

ec_pa_genes <- logfc_morph %>%
    filter(stage == "early_spermatocyte" & p_val_adj < 0.01 & avg_log2FC < 0) %>%
    pull("gene")
lc_pa_genes <- logfc_morph %>%
    filter(stage == "late_spermatocyte" & p_val_adj < 0.01 & avg_log2FC < 0) %>%
    pull("gene")
s_pa_genes <- logfc_morph %>%
    filter(stage == "late_spermatocyte" & p_val_adj < 0.01 & avg_log2FC < 0) %>%
    pull("gene")

In [65]:
length(ec_eu_genes)
length(lc_eu_genes)
length(s_eu_genes)
length(ec_pa_genes)
length(lc_pa_genes)
length(s_pa_genes)

[1] 4377

[1] 4023

[1] 4023

[1] 5543

[1] 4531

[1] 4531

## Investigate mutant genes in single cell data set

### Look at expression patterns of the genes of interest

In [66]:
## identify the presence of mutant genes 
potential_interest %>%
    rename(gene = symbol) %>%
    left_join(logfc_morph, by = "gene")

gene,position,change,consequence,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,stage
<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
LOC6901779,645002,-/AGCCAC,protein_altering_variant,1.355272e-42,-0.8570507,0.158,0.170,1.949017e-38,early_spermatocyte
LOC4815337,893839,G/C,stop_gained,NA,NA,NA,NA,NA,NA
LOC6902117,1625489,-/TGT,protein_altering_variant,5.042512e-18,-0.2748630,0.106,0.084,7.251636e-14,early_spermatocyte
LOC4815785,2706647,G/T,stop_gained,1.307539e-29,-0.7183186,0.156,0.162,1.880372e-25,early_spermatocyte
LOC4815774,2949241,C/T,stop_gained,NA,NA,NA,NA,NA,NA
LOC4815426,3468139,-/ACCAGT,protein_altering_variant,5.583518e-21,0.1295619,0.068,0.038,8.029657e-17,late_spermatocyte
LOC6902257,4221843,-/TGCTGC,protein_altering_variant,1.209741e-81,0.3858045,0.632,0.430,1.739729e-77,late_spermatocyte
LOC4815659,4250261,C/A,stop_gained,1.655317e-60,0.2096326,0.360,0.229,2.380511e-56,early_spermatocyte
LOC26532021,4393675,T/G,stop_gained,1.043152e-15,1.0007632,0.036,0.010,1.500156e-11,early_spermatocyte


### Look at expression pattern of differentially expressed genes for mutant

In [67]:
## add single cell data in annotated logfc
full_annotated_logfc <- annotated_logfc %>%
    mutate(Eusperm_Early_Spermatocytes = case_when(gene %in% ec_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Eusperm_Late_Spermatocytes = case_when(gene %in% lc_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Eusperm_Spermatids = case_when(gene %in% s_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Early_Spermatocytes = case_when(gene %in% ec_pa_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Late_Spermatocytes = case_when(gene %in% lc_pa_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Spermatids = case_when(gene %in% s_pa_genes ~ "deg",
                                 TRUE ~ "not deg")) %>%
    rename(dpse = gene) %>%
    left_join(mel_ortho, by = "dpse") %>%
    unique() %>%
    rename(gene = dpse)
write_csv(full_annotated_logfc, "intermediate/full_annotated_logfc.csv")

In [68]:
## look for enrichment of downregulated genes for each stage
ec_eu_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Eusperm_Early_Spermatocytes, direction) %>% unique() %>%
    group_by(Eusperm_Early_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Early_Spermatocytes") %>%
    rename(not_significant = 2)
ec_eu_down_chi <- ec_eu_down_df %>%
    as.matrix() %>%
    chisq.test()

ec_pa_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Parasperm_Early_Spermatocytes, direction) %>% unique() %>%
    group_by(Parasperm_Early_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Early_Spermatocytes") %>%
    rename(not_significant = 2)
ec_pa_down_chi <- ec_pa_down_df %>%
    as.matrix() %>%
    chisq.test()

lc_eu_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Eusperm_Late_Spermatocytes, direction) %>% unique() %>%
    group_by(Eusperm_Late_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Late_Spermatocytes") %>%
    rename(not_significant = 2)
lc_eu_down_chi <- lc_eu_down_df %>%
    as.matrix() %>%
    chisq.test()
    
lc_pa_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Parasperm_Late_Spermatocytes, direction) %>% unique() %>%
    group_by(Parasperm_Late_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Late_Spermatocytes") %>%
    rename(not_significant = 2)
lc_pa_down_chi <- lc_pa_down_df %>%
    as.matrix() %>%
    chisq.test()

s_eu_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Eusperm_Spermatids, direction) %>% unique() %>%
    group_by(Eusperm_Spermatids, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Spermatids") %>%
    rename(not_significant = 2)
s_eu_down_chi <- s_eu_down_df %>%
    as.matrix() %>%
    chisq.test()

s_pa_down_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hyper") %>%    
    select(gene, Parasperm_Spermatids, direction) %>% unique() %>%
    group_by(Parasperm_Spermatids, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Spermatids") %>%
    rename(not_significant = 2)
s_pa_down_chi <- s_pa_down_df %>%
    as.matrix() %>%
    chisq.test()

## look for enrichment of upregulated genes for each stage
ec_eu_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Eusperm_Early_Spermatocytes, direction) %>% unique() %>%
    group_by(Eusperm_Early_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Early_Spermatocytes") %>%
    rename(not_significant = 2)
ec_eu_up_chi <- ec_eu_up_df %>%
    as.matrix() %>%
    chisq.test()

ec_pa_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Parasperm_Early_Spermatocytes, direction) %>% unique() %>%
    group_by(Parasperm_Early_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Early_Spermatocytes") %>%
    rename(not_significant = 2)
ec_pa_up_chi <- ec_pa_up_df %>%
    as.matrix() %>%
    chisq.test()

lc_eu_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Eusperm_Late_Spermatocytes, direction) %>% unique() %>%
    group_by(Eusperm_Late_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Late_Spermatocytes") %>%
    rename(not_significant = 2)
lc_eu_up_chi <- lc_eu_up_df %>%
    as.matrix() %>%
    chisq.test()

lc_pa_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Parasperm_Late_Spermatocytes, direction) %>% unique() %>%
    group_by(Parasperm_Late_Spermatocytes, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Late_Spermatocytes") %>%
    rename(not_significant = 2)
lc_pa_up_chi <- lc_pa_up_df %>%
    as.matrix() %>%
    chisq.test()

s_eu_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Eusperm_Spermatids, direction) %>% unique() %>%
    group_by(Eusperm_Spermatids, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Eusperm_Spermatids") %>%
    rename(not_significant = 2)
s_eu_up_chi <- s_eu_up_df %>%
    as.matrix() %>%
    chisq.test()

s_pa_up_df <- full_annotated_logfc %>%
    filter(comparison == "sex ratio_mutant" & direction != "hypo") %>%    
    select(gene, Parasperm_Spermatids, direction) %>% unique() %>%
    group_by(Parasperm_Spermatids, direction) %>%
    count() %>%
    pivot_wider(names_from = direction, values_from = n) %>%
    column_to_rownames("Parasperm_Spermatids") %>%
    rename(not_significant = 2)
s_pa_up_chi <- s_pa_up_df %>%
    as.matrix() %>%
    chisq.test()

In [69]:
## create data frame of enrichment
morph_mutant_enrichment <- data.frame(stage = c("early_eusperm", "early_eusperm", "early_parasperm", "early_parasperm", 
                                                "late_eusperm", "late_eusperm", "late_parasperm", "late_parasperm", 
                                                "eusperm_spermatids", "eusperm_spermatids", "parasperm_spermatids", "parasperm_spermatids"),
                                      direction = rep(c("down", "up"), times = 6),
                                      pvalue = c(ec_eu_down_chi$p.value, ec_eu_up_chi$p.value, ec_pa_down_chi$p.value, ec_pa_up_chi$p.value,
                                                 lc_eu_down_chi$p.value, lc_eu_up_chi$p.value, lc_pa_down_chi$p.value, lc_pa_up_chi$p.value,
                                                 s_eu_down_chi$p.value, s_eu_up_chi$p.value, s_pa_down_chi$p.value, s_pa_up_chi$p.value),
                                      deg_ratio = c(ec_eu_down_df["deg", "hypo"]/ec_eu_down_df["deg", "not_significant"], ec_eu_up_df["deg", "hyper"]/ec_eu_up_df["deg", "not_significant"],
                                                    ec_pa_down_df["deg", "hypo"]/ec_pa_down_df["deg", "not_significant"], ec_pa_up_df["deg", "hyper"]/ec_pa_up_df["deg", "not_significant"],
                                                    lc_eu_down_df["deg", "hypo"]/lc_eu_down_df["deg", "not_significant"], lc_eu_up_df["deg", "hyper"]/lc_eu_up_df["deg", "not_significant"],
                                                    lc_pa_down_df["deg", "hypo"]/lc_pa_down_df["deg", "not_significant"], lc_pa_up_df["deg", "hyper"]/lc_pa_up_df["deg", "not_significant"],
                                                    s_eu_down_df["deg", "hypo"]/s_eu_down_df["deg", "not_significant"], s_eu_up_df["deg", "hyper"]/s_eu_up_df["deg", "not_significant"],
                                                    s_pa_down_df["deg", "hypo"]/s_pa_down_df["deg", "not_significant"], s_pa_up_df["deg", "hyper"]/s_pa_up_df["deg", "not_significant"]),
                                      notdeg_ratio = c(ec_eu_down_df["not deg", "hypo"]/ec_eu_down_df["not deg", "not_significant"], ec_eu_up_df["not deg", "hyper"]/ec_eu_up_df["not deg", "not_significant"],
                                                    ec_pa_down_df["not deg", "hypo"]/ec_pa_down_df["not deg", "not_significant"], ec_pa_up_df["not deg", "hyper"]/ec_pa_up_df["not deg", "not_significant"],
                                                    lc_eu_down_df["not deg", "hypo"]/lc_eu_down_df["not deg", "not_significant"], lc_eu_up_df["not deg", "hyper"]/lc_eu_up_df["not deg", "not_significant"],
                                                    lc_pa_down_df["not deg", "hypo"]/lc_pa_down_df["not deg", "not_significant"], lc_pa_up_df["not deg", "hyper"]/lc_pa_up_df["not deg", "not_significant"],
                                                    s_eu_down_df["not deg", "hypo"]/s_eu_down_df["not deg", "not_significant"], s_eu_up_df["not deg", "hyper"]/s_eu_up_df["not deg", "not_significant"],
                                                    s_pa_down_df["not deg", "hypo"]/s_pa_down_df["not deg", "not_significant"], s_pa_up_df["not deg", "hyper"]/s_pa_up_df["not deg", "not_significant"]))
## perform multiple testing correction
morph_mutant_enrichment$adjusted_p <- p.adjust(morph_mutant_enrichment$pvalue, method = "BH")

## interpret results of chi squared test
morph_mutant_enrichment %>%
    mutate(enrichment = case_when(adjusted_p < 0.01 & deg_ratio > notdeg_ratio ~ "over represented in stage",
                                  adjusted_p < 0.01 & deg_ratio < notdeg_ratio ~ "under represented in stage",
                                  TRUE ~ "not significant"))

stage,direction,pvalue,deg_ratio,notdeg_ratio,adjusted_p,enrichment
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
early_eusperm,down,9.965923e-06,0.7500000,0.5507442,2.989777e-05,over represented in stage
early_eusperm,up,2.301646e-04,0.4587500,0.6082544,5.523951e-04,under represented in stage
early_parasperm,down,7.159107e-06,0.5173745,0.7069243,2.863643e-05,under represented in stage
early_parasperm,up,6.624595e-04,0.6293436,0.4943639,1.165645e-03,over represented in stage
late_eusperm,down,1.221893e-02,0.3925234,0.6319668,1.466272e-02,not significant
late_eusperm,up,4.198307e-01,0.6355140,0.5518194,4.198307e-01,not significant
late_parasperm,down,7.770967e-04,0.8833333,0.5981888,1.165645e-03,over represented in stage
late_parasperm,up,1.785990e-07,0.2388889,0.5829361,1.071594e-06,under represented in stage
eusperm_spermatids,down,1.221893e-02,0.3925234,0.6319668,1.466272e-02,not significant


### Look at expression pattern of branch associated genes

In [70]:
## look for enrichment of downregulated genes for each stage
ec_eu_branch_df <- full_annotated_logfc %>%
    select(gene, Eusperm_Early_Spermatocytes, credible) %>% unique() %>%
    group_by(Eusperm_Early_Spermatocytes, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Eusperm_Early_Spermatocytes")
ec_eu_branch_chi <- ec_eu_branch_df %>%
    as.matrix() %>%
    chisq.test()

ec_pa_branch_df <- full_annotated_logfc %>%
    select(gene, Parasperm_Early_Spermatocytes, credible) %>% unique() %>%
    group_by(Parasperm_Early_Spermatocytes, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Parasperm_Early_Spermatocytes")
ec_pa_branch_chi <- ec_pa_branch_df %>%
    as.matrix() %>%
    chisq.test()

lc_eu_branch_df <- full_annotated_logfc %>% 
    select(gene, Eusperm_Late_Spermatocytes, credible) %>% unique() %>%
    group_by(Eusperm_Late_Spermatocytes, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Eusperm_Late_Spermatocytes")
lc_eu_branch_chi <- lc_eu_branch_df %>%
    as.matrix() %>%
    chisq.test()
    
lc_pa_branch_df <- full_annotated_logfc %>% 
    select(gene, Parasperm_Late_Spermatocytes, credible) %>% unique() %>%
    group_by(Parasperm_Late_Spermatocytes, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Parasperm_Late_Spermatocytes")
lc_pa_branch_chi <- lc_pa_branch_df %>%
    as.matrix() %>%
    chisq.test()

s_eu_branch_df <- full_annotated_logfc %>%
    select(gene, Eusperm_Spermatids, credible) %>% unique() %>%
    group_by(Eusperm_Spermatids, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Eusperm_Spermatids")
s_eu_branch_chi <- s_eu_branch_df %>%
    as.matrix() %>%
    chisq.test()

s_pa_branch_df <- full_annotated_logfc %>%
    select(gene, Parasperm_Spermatids, credible) %>% unique() %>%
    group_by(Parasperm_Spermatids, credible) %>%
    count() %>%
    pivot_wider(names_from = credible, values_from = n) %>%
    column_to_rownames("Parasperm_Spermatids")
s_pa_branch_chi <- s_pa_branch_df %>%
    as.matrix() %>%
    chisq.test()

In [71]:
## create data frame of enrichment
morph_branch_enrichment <- data.frame(stage = c("early_eusperm", "early_parasperm", "late_eusperm", "late_parasperm", "eusperm_spermatids", "parasperm_spermatids"),
                                      pvalue = c(ec_eu_branch_chi$p.value, ec_pa_branch_chi$p.value,
                                                 lc_eu_branch_chi$p.value, lc_pa_branch_chi$p.value,
                                                 s_eu_branch_chi$p.value, s_pa_branch_chi$p.value),
                                      deg_ratio = c(ec_eu_branch_df["deg", "credible"]/ec_eu_branch_df["deg", "non-credible"], ec_pa_branch_df["deg", "credible"]/ec_pa_branch_df["deg", "non-credible"],
                                                    lc_eu_branch_df["deg", "credible"]/lc_eu_branch_df["deg", "non-credible"], lc_pa_branch_df["deg", "credible"]/lc_pa_branch_df["deg", "non-credible"],
                                                    s_eu_branch_df["deg", "credible"]/s_eu_branch_df["deg", "non-credible"], s_pa_branch_df["deg", "credible"]/s_pa_branch_df["deg", "non-credible"]),
                                      notdeg_ratio = c(ec_eu_branch_df["not deg", "credible"]/ec_eu_branch_df["not deg", "non-credible"], ec_pa_branch_df["not deg", "credible"]/ec_pa_branch_df["not deg", "non-credible"],
                                                    lc_eu_branch_df["not deg", "credible"]/lc_eu_branch_df["not deg", "non-credible"], lc_pa_branch_df["not deg", "credible"]/lc_pa_branch_df["not deg", "non-credible"],
                                                    s_eu_branch_df["not deg", "credible"]/s_eu_branch_df["not deg", "non-credible"], s_pa_branch_df["not deg", "credible"]/s_pa_branch_df["not deg", "non-credible"]))
## perform multiple testing correction
morph_branch_enrichment$adjusted_p <- p.adjust(morph_branch_enrichment$pvalue, method = "BH")

## interpret results of chi squared test
morph_branch_enrichment %>%
    mutate(enrichment = case_when(adjusted_p < 0.01 & deg_ratio > notdeg_ratio ~ "over represented in stage",
                                  adjusted_p < 0.01 & deg_ratio < notdeg_ratio ~ "under represented in stage",
                                  TRUE ~ "not significant"))

stage,pvalue,deg_ratio,notdeg_ratio,adjusted_p,enrichment
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
early_eusperm,2.509364e-09,0.11202014,0.05802387,1.505618e-08,over represented in stage
early_parasperm,1.123793e-06,0.05402844,0.09579158,3.371380e-06,under represented in stage
late_eusperm,5.986044e-01,0.06372549,0.07725517,6.334920e-01,not significant
late_parasperm,6.334920e-01,0.08522727,0.07594639,6.334920e-01,not significant
eusperm_spermatids,5.986044e-01,0.06372549,0.07725517,6.334920e-01,not significant
parasperm_spermatids,6.334920e-01,0.08522727,0.07594639,6.334920e-01,not significant


In [72]:
## get the list of genes whose increase in expression is associated with an increase in parasperm content
positive_branch_genes <- credible_df %>%
    filter(sign > 0) %>%
    pull(transcript_id)

## determine what these positive branch genes are doing in the single cell data
logfc_morph %>%
    filter(gene %in% positive_branch_genes & p_val_adj < 0.01) %>%
    mutate(direction = case_when(avg_log2FC < 0 ~ "eusperm",
                                 avg_log2FC > 0 ~ "parasperm")) %>%
    group_by(stage, direction) %>%
    count()

stage,direction,n
<chr>,<chr>,<int>
early_spermatocyte,eusperm,4
early_spermatocyte,parasperm,6
late_spermatocyte,eusperm,2


In [121]:
logfc_morph %>%
    filter(gene %in% positive_branch_genes & p_val_adj < 0.01) %>%
    mutate(direction = case_when(avg_log2FC < 0 ~ "eusperm",
                                 avg_log2FC > 0 ~ "parasperm"))

gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,stage,direction
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
LOC6897142,0.000000e+00,2.1381177,0.991,0.789,0.000000e+00,early_spermatocyte,parasperm
LOC6897697,0.000000e+00,1.3177060,0.974,0.802,0.000000e+00,early_spermatocyte,parasperm
LOC4818038,0.000000e+00,0.7866458,0.985,0.886,0.000000e+00,early_spermatocyte,parasperm
LOC4803583,0.000000e+00,2.2592401,1.000,0.982,0.000000e+00,early_spermatocyte,parasperm
LOC4818032,2.527760e-70,0.3207878,0.965,0.888,3.635172e-66,early_spermatocyte,parasperm
LOC4803901,4.662660e-58,-0.2823437,0.997,0.997,6.705371e-54,early_spermatocyte,eusperm
LOC6897264,1.149386e-54,-0.2460309,0.999,1.000,1.652932e-50,early_spermatocyte,eusperm
LOC6902046,1.027001e-38,0.1807452,0.967,0.917,1.476930e-34,early_spermatocyte,parasperm
LOC4818036,5.779565e-13,-0.1245023,0.955,0.948,8.311592e-09,early_spermatocyte,eusperm


## Characterize Genes of Interest

In [75]:
## get the genes with the interesting mutations
potential_genes <- potential_interest %>%
    pull("symbol")

annotated_logfc %>%
    filter(gene %in% potential_genes) %>%
    rename(symbol = gene) %>%
    select(symbol, direction) %>%
    right_join(potential_interest, by = "symbol") %>%
    mutate(Eusperm_Early_Spermatocytes = case_when(symbol %in% ec_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Eusperm_Late_Spermatocytes = case_when(symbol %in% lc_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Eusperm_Spermatids = case_when(symbol %in% s_eu_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Early_Spermatocytes = case_when(symbol %in% ec_pa_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Late_Spermatocytes = case_when(symbol %in% lc_pa_genes ~ "deg",
                                 TRUE ~ "not deg"),
           Parasperm_Spermatids = case_when(symbol %in% s_pa_genes ~ "deg",
                                 TRUE ~ "not deg"),
           credible = case_when(symbol %in% credible_down_genes ~ "credible"),
           testes = case_when(symbol %in% testes_genes ~ "testes")) %>%
    rename(dpse = symbol) %>%
    left_join(mel_ortho, by = "dpse") %>%
    unique()

,dpse,direction,position,change,consequence,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,credible,testes,dmel,paralog,score
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,LOC4814348,not significant,6196915,C/A,stop_gained,not deg,not deg,not deg,deg,not deg,not deg,NA,NA,CG18259,1:m,NA
2,LOC4814348,not significant,6196915,C/A,stop_gained,not deg,not deg,not deg,deg,not deg,not deg,NA,NA,CG6961,1:m,30805
3,LOC6901779,hyper,645002,-/AGCCAC,protein_altering_variant,not deg,not deg,not deg,deg,not deg,not deg,NA,NA,CHES-1-like,1:1,12587
4,LOC6902257,hyper,4221843,-/TGCTGC,protein_altering_variant,not deg,deg,deg,not deg,not deg,not deg,NA,NA,HUWE1,1:1,12248
5,LOC4815337,NA,893839,G/C,stop_gained,not deg,not deg,not deg,not deg,not deg,not deg,NA,NA,Nek2,1:1,25920
6,LOC6902117,NA,1625489,-/TGT,protein_altering_variant,not deg,not deg,not deg,deg,not deg,not deg,NA,NA,CG14200,1:1,12375
7,LOC4815785,NA,2706647,G/T,stop_gained,not deg,not deg,not deg,deg,not deg,not deg,NA,NA,C3G,1:1,25901
8,LOC4815774,NA,2949241,C/T,stop_gained,not deg,not deg,not deg,not deg,not deg,not deg,NA,NA,CG9673,1:1,14705
9,LOC4815426,NA,3468139,-/ACCAGT,protein_altering_variant,not deg,deg,deg,not deg,not deg,not deg,NA,NA,Met,1:1,12860


## Investigate specific genes that would be of interest based on previous literature

### Set up Data

In [76]:
## load in orthologs between D. melanogaster and D. pseudoobscura
mel_ortho <- read_delim("../workflow/trinotate/drosoma_pse_mel.txt", col_names = FALSE)
names(mel_ortho) <- c("dpse", "dmel", "paralog", "score")

Rows: 13772 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): X1, X2, X3
dbl (1): X4

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [100]:
## load in teh list of SFPs from Wigby et al. 2020
wigby_sfp <- read_csv("input/wigby_2020_sfp_list.csv", skip = 1) %>%
    janitor::clean_names()

Rows: 713 Columns: 29
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (27): Name, Symbol, CG#, FBgn#, Cytogenetic map, Chromosome, Strand, Pro...
dbl  (2): Location Max, Location Min

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Search data for specific genes

In [119]:
## identify D. pseudoobscura paralogs
test <- wigby_sfp %>%
    rename(dmel = symbol) %>%
    inner_join(mel_ortho, by = "dmel", relationship = "many-to-many") %>%
    select(name, dpse, chromosome) %>%
    rename(gene = dpse)

## determine if SFPs appear in our data set as differentially expressed are associated with parasperm
annotated_sfp <- full_annotated_logfc %>%
    inner_join(test, by = "gene", relationship = "many-to-many") %>%
    filter(comparison == "sex ratio_mutant")

annotated_sfp %>%
    select(gene, direction) %>% unique() %>%
    group_by(direction) %>%
    count()
annotated_sfp %>%
    select(gene, credible) %>% unique() %>%
    group_by(credible) %>%
    count()

direction,n
<chr>,<int>
hyper,21
hypo,8
not significant,18


credible,n
<chr>,<int>
non-credible,47


In [135]:
annotated_sfp_trunc <- annotated_sfp %>%
    select(gene, dmel, name, logfc, pvalue) %>%
    left_join(tau, by = "gene") %>%
    mutate(early_spermatocyte = case_when(gene %in% ec_eu_genes ~ "eusperm",
                                          gene %in% ec_pa_genes ~ "parasperm",
                                          TRUE ~ "not_deg"),
           late_spermatocyte = case_when(gene %in% lc_eu_genes ~ "eusperm",
                                         gene %in% lc_pa_genes ~ "parasperm",
                                         TRUE ~ "not_deg"),
           spermatid = case_when(gene %in% s_eu_genes ~ "eusperm",
                                 gene %in% s_pa_genes ~ "parasperm",
                                 TRUE ~ "not_deg")) %>%
    select(gene, dmel, logfc, pvalue, tau, tau_tissue, early_spermatocyte, late_spermatocyte, spermatid) %>%
    rename(dpse_name = gene, dmel_name = dmel) %>%
    unique()
# write_csv(annotated_sfp_trunc, "intermediate/pseudoobscura_heteromorphism_sfp.csv")

In [115]:
## get the list of D. pseudoobscura orthologs of SFPs in our dataset
dpse_sfp <- unique(annotated_sfp$gene)

length(dpse_sfp)
length(dpse_sfp[dpse_sfp %in% ec_eu_genes])
length(dpse_sfp[dpse_sfp %in% lc_eu_genes])
length(dpse_sfp[dpse_sfp %in% s_eu_genes])
length(dpse_sfp[dpse_sfp %in% ec_pa_genes])
length(dpse_sfp[dpse_sfp %in% lc_pa_genes])
length(dpse_sfp[dpse_sfp %in% s_pa_genes])

[1] 47

[1] 17

[1] 1

[1] 1

[1] 18

[1] 3

[1] 3

In [120]:
## determine if any sperm proteins are in the data set
full_annotated_logfc %>%
    rename(symbol = gene) %>%
    inner_join(sperm_dpse, by = "symbol") %>%
    filter(comparison == "sex ratio_mutant")

symbol,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,⋯,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score,sperm_protein
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
LOC4817494,17783,sex ratio_mutant,0.03738714,9.998863e-01,not significant,not significant,not significant,non-credible,other,⋯,not deg,not deg,not deg,deg,not deg,not deg,Idgf3,1:1,5717,Idgf3
LOC4817554,17918,sex ratio_mutant,-0.34200737,6.013668e-03,significant,hypo,marginally differentially expressed,non-credible,other,⋯,deg,not deg,not deg,not deg,not deg,not deg,Pde1c,1:1,5446,Phosphodiesterase 1c
LOC6901019,18869,sex ratio_mutant,1.62654994,4.745812e-05,significant,hyper,marginally differentially expressed,non-credible,other,⋯,not deg,not deg,not deg,not deg,not deg,not deg,Tsf1,1:1,19710,Transferrin 1
LOC4814222,19001,sex ratio_mutant,1.75668576,2.190587e-04,significant,hyper,marginally differentially expressed,non-credible,other,⋯,deg,not deg,not deg,not deg,not deg,not deg,Idgf4,1:1,19197,Imaginal disc growth factor 4
LOC26532861,6593,sex ratio_mutant,-0.50714680,7.927122e-05,significant,hypo,marginally differentially expressed,non-credible,testes,⋯,not deg,not deg,not deg,deg,not deg,not deg,scpr-C,n:m,32268,SCP-containing protein C
LOC26532861,6593,sex ratio_mutant,-0.50714680,7.927122e-05,significant,hypo,marginally differentially expressed,non-credible,testes,⋯,not deg,not deg,not deg,deg,not deg,not deg,scpr-B,n:m,NA,SCP-containing protein C
LOC26532861,6593,sex ratio_mutant,-0.50714680,7.927122e-05,significant,hypo,marginally differentially expressed,non-credible,testes,⋯,not deg,not deg,not deg,deg,not deg,not deg,scpr-A,n:m,NA,SCP-containing protein C
LOC4803148,6605,sex ratio_mutant,1.01286730,8.833909e-09,significant,hyper,marginally differentially expressed,non-credible,testes,⋯,not deg,not deg,not deg,deg,not deg,not deg,scpr-C,n:m,NA,SCP-containing protein C
LOC4803148,6605,sex ratio_mutant,1.01286730,8.833909e-09,significant,hyper,marginally differentially expressed,non-credible,testes,⋯,not deg,not deg,not deg,deg,not deg,not deg,scpr-B,n:m,NA,SCP-containing protein C


In [140]:
mel_ortho %>%
    filter(dmel == "vis" | dmel == "CROL" | dmel == "Caup" ) %>%
    rename(gene = dpse) %>%
    left_join(full_annotated_logfc, by = "gene")

gene,dmel.x,paralog.x,score.x,...1,comparison,logfc,pvalue,significance,direction,⋯,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel.y,paralog.y,score.y
<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC4805467,vis,1:m,NA,11834,sex ratio_mutant,0.2575892,0.3066086,not significant,not significant,⋯,other,deg,not deg,not deg,not deg,not deg,not deg,vis,1:m,NA
LOC4805467,vis,1:m,NA,11834,sex ratio_mutant,0.2575892,0.3066086,not significant,not significant,⋯,other,deg,not deg,not deg,not deg,not deg,not deg,achi,1:m,6728


In [90]:
## nebenkern formation
## fzo (fuzzy onion) - not found (first gene related to mitochondria, cannot aggregate)
## Larp - not found (partitions mitochondria into nebenkern)
## Pink1 (LOC4815974) - not differential (maintains separation)
## Parkin - not found (maintains separation) -> tried park (uniprot), not differential
test <- mel_ortho %>%
    filter(dmel == "park") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
LOC4812935,26105,sex ratio_mutant,-0.2605455,0.2726715,not significant,not significant,not significant,non-credible,other,other,deg,not deg,not deg,not deg,not deg,not deg,park,1:1,20550


In [91]:
## nebenkern unfurling
## Milton - not found (anchors nebernkern to nuclei and allows sliding of microtubules) -> tried milt (uniprot), not significant
## Lis-1 (LOC4803869) - not differential (anchors nebenkern to nuclei)
## Miro (LOC4801756) - not differential (allows sliding of cytosolic microtubules)
## Nebbish - not found (elongation is truncated) -> tried neb (uniprot), not significant
## Khc73 - not found (elongation is truncated) -> tried Khc-73 (uniprot), significant
## fascetto - not found (elongation is truncated)  -> tried feo (uniprot), significant
## nmd (LOC117183914) - not differential (decreased number of mitochondria)
## cnn (LOC4804379) - not differential (required for assembly of flagellar axoneme)
## Grip members - 3/7 differentially expressed (localize to basal body for microtubule formation)
test <- mel_ortho %>%
    filter(dmel == "feo") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


In [92]:
## mitochondrial energetics
## Mic26-27 - not found (part of cristae organizing system)
## QIL1 (LOC4812916) - not differential (part of cristae organizing system)
## Mitofilin (LOC6896986) - not differential (part of cristae organizing system)
## Chchd3 (LOC4800334) - not differential (part of cristae organizing system)
## knon (LOC4803786) - not differential (electron transportation)
## knon (LOC4803786) - not differential (electron transportation)
## blw (LOC4803853) - differential (electron transportation)
test <- mel_ortho %>%
    filter(dmel == "Mic26-27") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


In [82]:
## list of genes with differential expression
mito_genes <- c("Khc-73", "feo", "blw")
mel_ortho %>%
    filter(dmel %in% mito_genes | grepl("Grip", dmel)) %>%
    select(dpse, dmel) %>%
    rename(gene = dpse) %>%
    left_join(annotated_logfc, by = "gene") %>%
    select(-c(pvalue, direction, color))

gene,dmel,...1,comparison,logfc,significance,credible,testes_specificity,x
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
LOC4803853,blw,8243,sex ratio_mutant,0.4075890,significant,non-credible,other,other
LOC4804863,Khc-73,10481,sex ratio_mutant,0.4705689,significant,non-credible,other,other
LOC6899804,Grip75,NA,NA,NA,NA,NA,NA,NA
LOC4817648,Grip71,NA,NA,NA,NA,NA,NA,NA
LOC6902241,feo,NA,NA,NA,NA,NA,NA,NA
LOC4815237,Grip84,NA,NA,NA,NA,NA,NA,NA
LOC4815390,Grip,NA,NA,NA,NA,NA,NA,NA
LOC4815196,Grip91,22097,sex ratio_mutant,-0.1469523,not significant,non-credible,other,other
LOC6901874,Grip128,NA,NA,NA,NA,NA,NA,NA


In [93]:
## acrosome genes
## Four way stop (fws) - not differential (golgi and endosomal trafficking regulators)
## Giotto (Gio) - not found (golgi and endosomal trafficking regulators)
## Rab11 - not differential (golgi and endosomal trafficking regulators)
## Nuclear fallout (nuf) - not differential (Rab11 effector Nuclear fallout)
## Brunelleschi (brun) - differential (golgi and endosomal trafficking regulators)
## Lectin Wheat Germ Agglutinin (WGA) - not found (secretory trafficking)
## Sneaky (snky) - differential (sperm plasma membrane breakdown during fertilization)
## Misfire (mfr) - differential (sperm plasma membrane breakdown)
test <- mel_ortho %>%
    filter(dmel == "mfr") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


In [94]:
## golgi apparatus genes
## Golgins lava lamp (Lva) - differential
## GCC88 - not differential
## GM130 - not differential
## Golgi alpha-mannosidase II (GMII) - not found; (alpha-Man-IIa) - differential
## Golgi phosphoprotein-3 (GOLPHh3) - not found; (sau) - not differential
## Coatomer (COP I) - not found; (gammaCOP) - not significant
## Gga - not differential (clathrin adaptors)
## AP-1 - not found (clathrin adaptors)
test <- mel_ortho %>%
    filter(dmel == "AP-1") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


In [95]:
## search for aquaporins
test <- mel_ortho %>%
    filter(grepl("AQP", dmel)) %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


In [97]:
## Odarant binding genes (Obp) - (odarant binding genes essential to male fertility)
test <- mel_ortho %>%
    filter(dmel == "lush") %>%
    pull("dpse")
full_annotated_logfc %>%
    filter(gene %in% test)
test <- mel_ortho %>%
    filter(grepl("Obp", dmel)) %>%
    rename(gene = dpse)
full_annotated_logfc %>%
    right_join(test, by = "gene")

gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,x,Eusperm_Early_Spermatocytes,Eusperm_Late_Spermatocytes,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel,paralog,score
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>


gene,...1,comparison,logfc,pvalue,significance,direction,color,credible,testes_specificity,⋯,Eusperm_Spermatids,Parasperm_Early_Spermatocytes,Parasperm_Late_Spermatocytes,Parasperm_Spermatids,dmel.x,paralog.x,score.x,dmel.y,paralog.y,score.y
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
LOC4804338,9341,sex ratio_mutant,-0.79341126,5.597574e-09,significant,hypo,marginally differentially expressed,non-credible,other,⋯,not deg,deg,not deg,not deg,Obp44a,1:1,18154,Obp44a,1:1,18154
LOC6898273,8300,sex ratio_mutant,0.06703936,9.977918e-01,not significant,not significant,not significant,non-credible,testes,⋯,not deg,not deg,not deg,not deg,Obp50b,1:1,6996,Obp50b,1:1,6996
LOC4800495,566,sex ratio_mutant,-0.26816585,2.086075e-01,not significant,not significant,not significant,non-credible,other,⋯,not deg,not deg,not deg,not deg,Obp99a,1:1,18152,Obp99a,1:1,18152
LOC4800502,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp99b,1:1,28366
LOC4800501,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp99d,1:1,18156
LOC6896778,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp99c,1:1,35000
LOC4801915,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp83cd,1:1,10453
LOC4801921,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp83b,1:1,24621
LOC4801922,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,Obp83a,1:1,24617
